# **Hito 1 · Baseline de recuperación**

**Procesamiento del Lenguaje Natural (UC3M, 2026/27)** · Chatbot sobre documentación de misiones espaciales

Este notebook contiene todo el Hito 1, que aplica a nuestro corpus lo visto en el notebook *Text Vectorization I*:

1. **Corpus:** 3.314 documentos sobre misiones espaciales.
2. **Fragmentación:** cada documento se divide en fragmentos de unas 300 palabras, que son la unidad de indexación.
3. **Preprocesado con spaCy:** lematización, filtrado por categoría gramatical, eliminación de stopwords y filtrado por frecuencia documental.
4. **Consultas de evaluación:** un LLM (ollama) escribe 100 consultas *known-item* a partir de documentos elegidos al azar; el equipo las revisa a mano y descarta las triviales o ambiguas. Las que quedan se guardan en `queries.jsonl`.
5. **Baselines:** TF-IDF con similitud coseno y BM25, con un ranking de documentos por consulta.

**Cómo ejecutarlo.** Desde la raíz del repositorio, con las dependencias de `requirements.txt` y el modelo `en_core_web_sm` instalados. La primera vez, la lematización con spaCy tarda unos 10 minutos y queda guardada en `cache/`; las siguientes, unos segundos. La generación de las consultas solo se hace una vez: el resultado se guarda en `consultas_generadas.jsonl`.

## **0. Configuración**

In [1]:
import os
import re
import json
import random
import hashlib
import zipfile
from getpass import getpass
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import urllib3
import spacy
import wordninja
import bm25s
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.models.phrases import Phrases
from gensim.similarities import SparseMatrixSimilarity

pd.set_option('display.max_colwidth', 100)

CORPUS_ZIP = Path('corpus/corpus_misiones_espaciales_json.zip')
CACHE_DIR = Path('cache')                              # lemas de spaCy (no se sube al repositorio)
GENERATED_PATH = Path('consultas_generadas.jsonl')     # las 100 consultas que escribió el LLM
QUERIES_PATH = Path('queries.jsonl')                   # las consultas que quedan tras la revisión

CONFIG = {
    # Fragmentación
    'chunk_words': 300,          # tamaño objetivo de un fragmento (palabras)
    'chunk_overlap': 45,         # solapamiento entre fragmentos consecutivos (15 %)
    'max_words_whole': 400,      # los documentos de hasta 400 palabras no se dividen
    # Preprocesado
    'spacy_model': 'en_core_web_sm',
    'valid_pos': ['NOUN', 'VERB', 'ADJ', 'PROPN'],
    'min_token_len': 2,
    'max_identifier_len': 4,     # se conservan identificadores con dígitos cortos: '2', '1969', 'sts' ...
    'specific_stopwords': ['et', 'al', 'fig', 'figure', 'table', 'page', 'ref', 'eq', 'etc'],
    'ngram_min_count': 5,
    'ngram_threshold': 20,
    'no_below': 2,               # un término debe aparecer en al menos 2 fragmentos
    'no_above': 0.5,             # y como mucho en la mitad de ellos
    # Consultas
    'n_queries': 100,
    'seed': 2026,
    'ollama_url': 'https://yiyuan.tsc.uc3m.es/api/generate',
    'ollama_model': 'llama3.1:8b',
    # Baselines
    'bm25_method': 'lucene',
    'bm25_k1': 1.5,
    'bm25_b': 0.75,
    'top_k': 10,
}

## **1. Corpus**

El corpus está en `corpus/corpus_misiones_espaciales_json.zip`: un JSON por documento (con `doc_id`, título, URL, fuente y texto completo) y un índice, `indice.json`. El fichero `LEEME.md` del ZIP describe las fuentes y la limpieza aplicada al extraerlas.

Comprobamos que todos los documentos del índice existen, tienen un `doc_id` único y contienen texto.

In [2]:
with zipfile.ZipFile(CORPUS_ZIP) as z:
    index = json.loads(z.read('indice.json'))
    docs = pd.DataFrame([json.loads(z.read(row['file'])) for row in index])
docs = docs[['doc_id', 'title', 'url', 'source', 'text']]

assert docs.doc_id.is_unique, 'Hay doc_id repetidos'
assert docs.doc_id.tolist() == [row['doc_id'] for row in index]
assert docs.text.str.strip().str.len().gt(0).all(), 'Hay documentos sin texto'
docs['words'] = docs.text.str.split().str.len()

print(f'{len(docs):,} documentos')
docs.groupby('source').agg(documentos=('doc_id', 'size'), palabras_mediana=('words', 'median'), palabras_max=('words', 'max'))

3,314 documentos


,documentos,palabras_mediana,palabras_max
source,,,
Apollo press kits,9,71397.0,112453
JPL PDFs,63,11539.0,36020
NTRS pilot,85,4852.0,292971
Wikipedia 1957–1999,1563,479.0,20459
Wikipedia 2000–2025,1594,443.5,14081


## **2. Fragmentación**

Los documentos van desde fichas de Wikipedia de unas decenas de palabras hasta *press kits* de cientos de páginas, así que no se pueden indexar enteros. Siguiendo las indicaciones del Hito 1:

* Los documentos de hasta 400 palabras se indexan **completos**.
* Los más largos se dividen en fragmentos de **200 a 300 palabras**, con un **solapamiento de 45 palabras** (15 %). El número de fragmentos se calcula para que todos tengan casi el mismo tamaño.
* Cada fragmento conserva el identificador de su documento y tiene uno propio: `doc_0123__0007`.

La unidad de indexación es el fragmento, así que las estadísticas de TF-IDF y BM25 (IDF, longitud media) se calculan sobre fragmentos.

In [3]:
WORD_RE = re.compile(r"\b\w+(?:['’-]\w+)*\b")

def split_document(text):
    # Devuelve el texto de cada fragmento, respetando el texto original (saltos de línea incluidos)
    spans = [m.span() for m in WORD_RE.finditer(text)]
    n = len(spans)
    if n <= CONFIG['max_words_whole']:
        return [text.strip()]
    step = CONFIG['chunk_words'] - CONFIG['chunk_overlap']
    k = int(np.ceil((n - CONFIG['chunk_overlap']) / step))        # nº de fragmentos
    size = (n + (k - 1) * CONFIG['chunk_overlap']) / k            # palabras por fragmento (entre 200 y 300)
    starts = [round(i * (size - CONFIG['chunk_overlap'])) for i in range(k)]
    ends = [round(s + size) for s in starts[:-1]] + [n]
    return [text[spans[s][0]:spans[e - 1][1]] for s, e in zip(starts, ends)]

rows = []
for doc in docs.itertuples():
    for i, chunk in enumerate(split_document(doc.text), start=1):
        rows.append({'chunk_id': f'{doc.doc_id}__{i:04d}', 'doc_id': doc.doc_id, 'text': chunk})
frags = pd.DataFrame(rows)
frags['words'] = frags.text.map(lambda t: len(WORD_RE.findall(t)))

assert frags.chunk_id.is_unique and set(frags.doc_id) == set(docs.doc_id)
long_docs = frags[frags.doc_id.map(frags.doc_id.value_counts()) > 1]
print(f'{len(frags):,} fragmentos de {frags.doc_id.nunique():,} documentos')
print(f'Fragmentos de documentos divididos: entre {long_docs.words.min()} y {long_docs.words.max()} palabras')
print('Documentos con más fragmentos:')
docs.set_index('doc_id').title[frags.doc_id.value_counts().head(5).index].to_frame().assign(fragmentos=frags.doc_id.value_counts().head(5))

19,620 fragmentos de 3,314 documentos
Fragmentos de documentos divididos: entre 215 y 300 palabras
Documentos con más fragmentos:


,title,fragmentos
doc_id,,
doc_3258,"On Mars: Exploration of the Red Planet, 1958-1978",1244
doc_3224,Apollo 11 Press Kit,433
doc_3267,Parker Solar Probe: Four Years of Discoveries at Solar Cycle Minimum,292
doc_3227,Apollo 14 Press Kit,281
doc_3229,Apollo 16 Press Kit,275


## **3. Preprocesado con spaCy**

Como en las secciones 1 y 2 de *Text Vectorization I*:

1. **Limpieza (*wrangling*):** se quitan URLs, se unen las palabras cortadas con guion al final de línea, se separan identificadores como `STS-129` o `2000/2001`, y se arregla el OCR de los PDF antiguos, que separa las letras (`s t r u c t u r e` → `structure`, con `wordninja`).
2. **spaCy** (`en_core_web_sm`): lematización y **filtrado por categoría gramatical** (sustantivos, verbos, adjetivos y nombres propios), eliminación de **stopwords** (las de spaCy y unas cuantas de documentos técnicos, como *fig* o *et al*) y de lemas de una letra. Se conservan los números e identificadores cortos (`1969`, `2`, `sts`), que son importantes en este dominio.
3. **N-gramas** con `Phrases` de Gensim (`apollo_11`, `space_station`...).
4. **Filtrado por frecuencia documental** con `Dictionary.filter_extremes`: se eliminan los términos que aparecen en un solo fragmento o en más de la mitad.

Las consultas pasarán por **la misma función**, `preprocess`, con el mismo modelo de n-gramas y el mismo diccionario, para que se preprocesen exactamente igual que los documentos.

In [4]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
HYPHEN_BREAK_RE = re.compile(r'(\w)-[ \t]*\n\s*(\w)')
SPACED_LETTERS_RE = re.compile(r'\b(?:[A-Za-z] ){2,}[A-Za-z]\b')
ID_SEPARATOR_RE = re.compile(r'(?<=[A-Za-z0-9])[-/](?=[A-Za-z0-9])')

def wrangle_text(text):
    text = URL_RE.sub(' ', text)
    text = HYPHEN_BREAK_RE.sub(r'\1\2', text)
    text = SPACED_LETTERS_RE.sub(lambda m: ' '.join(wordninja.split(m.group(0).replace(' ', ''))), text)
    text = ID_SEPARATOR_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

nlp = spacy.load(CONFIG['spacy_model'], disable=['parser', 'ner'])
VALID_POS = set(CONFIG['valid_pos'])
STOPWORDS = set(nlp.Defaults.stop_words) | set(CONFIG['specific_stopwords'])

def normalize_token(tk):
    # Forma normalizada del token, o None si hay que descartarlo
    text = tk.lower_
    if text.isalnum() and any(c.isdigit() for c in text):                 # números e identificadores
        return text if len(text) <= CONFIG['max_identifier_len'] else None
    if not tk.is_alpha or tk.pos_ not in VALID_POS:
        return None
    lemma = tk.lemma_.lower()
    if len(lemma) < CONFIG['min_token_len'] or not lemma.isalpha() or lemma in STOPWORDS or text in STOPWORDS:
        return None
    return lemma

def lemmatize(texts):
    return [[t for t in map(normalize_token, doc) if t]
            for doc in nlp.pipe((wrangle_text(t) for t in texts), batch_size=128)]

example = frags[frags.text.str.contains(SPACED_LETTERS_RE)].iloc[0]
print('Texto original:', example.text[:300].replace('\n', ' '))
print('\nLimpio:       ', wrangle_text(example.text)[:300])
print('\nLemas:        ', lemmatize([example.text])[0][:30])

Texto original: MIMOSA  Organization | Academy of Sciences of the Czech Republic  Contractor | Space Devices  Mission Type | Atmospheric science  Launch | June 30, 2003 on Rockot  Launch site | Plesetsk Cosmodrome  Mission duration | 18 months (calculated)  Mass | 66 kg (146 lb)  Webpage  Orbital elements  Semi-maj

Limpio:        MIMOSA Organization | Academy of Sciences of the Czech Republic Contractor | Space Devices Mission Type | Atmospheric science Launch | June 30, 2003 on Rockot Launch site | Plesetsk Cosmodrome Mission duration | 18 months (calculated) Mass | 66 kg (146 lb) Webpage Orbital elements Semi major axis | 

Lemas:         ['mimosa', 'organization', 'academy', 'sciences', 'czech', 'republic', 'contractor', 'space', 'devices', 'mission', 'type', 'atmospheric', 'science', 'launch', 'june', '30', '2003', 'rockot', 'launch', 'site', 'plesetsk', 'cosmodrome', 'mission', 'duration', '18', 'month', 'calculate', 'mass', '66', 'kg']


La lematización de todos los fragmentos se guarda en `cache/`. Si cambian los fragmentos o los parámetros, se vuelve a calcular.

In [5]:
%%time
lemma_key = hashlib.sha256(json.dumps(
    [CONFIG['spacy_model'], CONFIG['valid_pos'], CONFIG['min_token_len'], CONFIG['max_identifier_len'],
     CONFIG['specific_stopwords'], spacy.__version__, frags.chunk_id.tolist(), frags.text.tolist()]).encode()).hexdigest()
cache_file = CACHE_DIR / f'lemas_{lemma_key[:16]}.json'

if cache_file.exists():
    lemmas = json.loads(cache_file.read_text(encoding='utf-8'))
    print('Lemas cargados de', cache_file)
else:
    lemmas = lemmatize(frags.text.tolist())
    CACHE_DIR.mkdir(exist_ok=True)
    cache_file.write_text(json.dumps(lemmas), encoding='utf-8')
    print('Lemas calculados y guardados en', cache_file)

Lemas calculados y guardados en cache\lemas_605048cf4c36798d.json
CPU times: total: 5min 45s
Wall time: 5min 47s


In [6]:
%%time
phrase_model = Phrases(lemmas, min_count=CONFIG['ngram_min_count'], threshold=CONFIG['ngram_threshold']).freeze()
with_ngrams = [phrase_model[doc] for doc in lemmas]

D = Dictionary(with_ngrams)
n_before = len(D)
D.filter_extremes(no_below=CONFIG['no_below'], no_above=CONFIG['no_above'], keep_n=None)

def preprocess(texts, from_lemmas=None):
    # Texto -> tokens. La misma función para fragmentos y consultas.
    token_lists = from_lemmas if from_lemmas is not None else lemmatize(texts)
    return [[t for t in phrase_model[tokens] if t in D.token2id] for tokens in token_lists]

frags['tokens'] = preprocess(None, from_lemmas=lemmas)
n_tokens = frags.tokens.str.len()
print(f'{len(phrase_model.phrasegrams):,} n-gramas, p. ej.:', [g for g, _ in Counter(t for d in with_ngrams for t in d if '_' in t).most_common(12)])
print(f'Vocabulario: {n_before:,} -> {len(D):,} términos tras el filtrado por frecuencia documental')
print(f'Tokens por fragmento: media {n_tokens.mean():.1f}, mediana {n_tokens.median():.0f}; fragmentos sin tokens: {(n_tokens == 0).sum()}')

16,784 n-gramas, p. ej.: ['spacecraft_property', 'orbital_parameter', 'cospar_id', 'reference_system', 'mission_duration', 'start_mission', 'solar_wind', 'space_shuttle', 'low_earth', 'magnetic_field', 'geocentric_regime', 'km_mi']
Vocabulario: 93,549 -> 57,422 términos tras el filtrado por frecuencia documental
Tokens por fragmento: media 131.3, mediana 137; fragmentos sin tokens: 0
CPU times: total: 12.5 s
Wall time: 12.7 s


## **4. Consultas de evaluación**

Siguiendo el enunciado y la sección 4.1 de *Text Vectorization I*:

1. Se eligen **100 documentos al azar** (semilla fija) y, de cada uno, **un fragmento al azar**.
2. Un **LLM** (`llama3.1:8b`, servidor ollama de la UC3M) escribe, para cada fragmento, la consulta que haría un usuario que buscara ese documento, **sin copiar sus frases**. Se usa el prompt del notebook de la asignatura, adaptado a nuestro corpus.
3. El **equipo revisa a mano** las 100 consultas y **descarta las triviales o ambiguas**. No se corrige ninguna.
4. Las consultas que quedan se guardan en `queries.jsonl`. El documento de origen es el relevante para cada consulta (`relevant_ids`), y también se guarda el fragmento de origen (`source_chunk`).

### *4.1. Selección al azar*

In [7]:
rng = random.Random(CONFIG['seed'])
chunks_by_doc = frags.groupby('doc_id').chunk_id.apply(list)
selected_docs = rng.sample(docs.doc_id.tolist(), CONFIG['n_queries'])
candidates = pd.DataFrame({'cid': [f'c{i:03d}' for i in range(1, CONFIG['n_queries'] + 1)],
                           'doc_id': selected_docs,
                           'chunk_id': [rng.choice(chunks_by_doc[d]) for d in selected_docs]})
candidates = (candidates.merge(docs[['doc_id', 'title', 'source']], on='doc_id')
              .merge(frags[['chunk_id', 'text']], on='chunk_id').sort_values('cid', ignore_index=True))
print(candidates.source.value_counts().to_string())

source
Wikipedia 1957–1999    60
Wikipedia 2000–2025    35
JPL PDFs                3
NTRS pilot              2


### *4.2. Generación con ollama*

Cada consulta se guarda en `consultas_generadas.jsonl` en cuanto se genera. Si el fichero ya tiene las 100 consultas, esta celda no vuelve a llamar al LLM, así que el resultado no cambia al reejecutar el notebook.

La clave de la API es la del enunciado (sección 7). Se lee de la variable de entorno `OLLAMA_API_KEY` o se pide al ejecutar la celda.

In [8]:
import os
os.environ['OLLAMA_API_KEY'] = "sk-alumnos-38b7bcfb08c3833e97814eadefe5b7a5"

PROMPT = '''You write search queries for a collection of documents about space missions.
Below is a fragment of the document titled "{title}". Write ONE search query (8 to 20 words)
that a person looking for THIS document could type if they did not remember its name.

Rules:
- Do not write the name, designation, number or acronym of the mission or spacecraft
  ("{title}" or any variant of it).
- Include two or three specific facts from the fragment that tell this document apart from
  others about similar missions: for example the year or date, the country or operator,
  the purpose, or a notable event.
- Do not use generic phrases such as "mission details", "specifications" or "orbital parameters".
- Use your own words: do not copy sentences from the fragment.

Example (a made-up satellite):
Fragment: "Kestrel-3 is a Norwegian weather satellite launched on 4 March 2011 by a Soyuz-2 rocket
from Plesetsk. It failed after six weeks because of a battery fault."
Bad query: Kestrel-3 mission details, launch date and orbital parameters
Good query: Norwegian weather satellite sent up from Plesetsk in March 2011 that died after six weeks due to a battery problem

Answer with the query only.
Fragment: "{text}"'''

def ask_llm(text, title, api_key, seed):
    response = requests.post(CONFIG['ollama_url'], headers={'X-API-KEY': api_key}, verify=False, timeout=300,
                             json={'model': CONFIG['ollama_model'], 'prompt': PROMPT.format(title=title, text=text),
                                   'stream': False, 'options': {'temperature': 0.3, 'seed': seed}})
    response.raise_for_status()
    lines = response.json()['response'].strip().splitlines()
    line = lines[0] if lines else ''                      # solo la primera línea
    return re.sub(r'^(good |search )?query:\s*', '', line, flags=re.IGNORECASE).strip(' "\'')   # sin 'Query:' ni comillas

def contains_title(query, title):
    # 'Kosmos 903' -> 'kosmos903': detecta el título aunque cambien espacios, guiones o mayúsculas
    normalize = lambda s: re.sub(r'[^a-z0-9]', '', s.lower())
    name = normalize(re.sub(r'\(.*?\)', '', title))
    return bool(name) and name in normalize(query)

generated = {}
if GENERATED_PATH.exists():
    generated = {r['cid']: r for r in map(json.loads, GENERATED_PATH.read_text(encoding='utf-8').splitlines())}

missing = candidates[~candidates.cid.isin(generated)]
if len(missing):
    urllib3.disable_warnings()                            # el servidor de la UC3M se usa con 'curl -k'
    api_key = os.environ.get('OLLAMA_API_KEY') or getpass('API key de ollama (sección 7 del enunciado): ')
    with GENERATED_PATH.open('a', encoding='utf-8', newline='\n') as f:
        for c in missing.itertuples():
            for attempt in range(3):                      # se repite si la consulta contiene el título
                query = ask_llm(c.text, c.title, api_key, seed=CONFIG['seed'] + attempt)
                if not contains_title(query, c.title):
                    break
            generated[c.cid] = {'cid': c.cid, 'chunk_id': c.chunk_id, 'query': query}
            f.write(json.dumps(generated[c.cid], ensure_ascii=False) + '\n')
            print(f'{c.cid} {c.title[:40]:40s} -> {query}')

candidates['query'] = candidates.cid.map(lambda c: generated[c]['query'])
print(f'{len(candidates)} consultas generadas con {CONFIG["ollama_model"]}')

100 consultas generadas con llama3.1:8b


### *4.3. Revisión manual*

Para cada consulta se muestra el documento, la consulta y el fragmento del que sale. El enunciado pide **descartar las consultas triviales o ambiguas**:

* **Trivial:** se responde sin esfuerzo porque repite lo que identifica al documento, por ejemplo el nombre de la misión o frases copiadas del texto.
* **Ambigua:** vale igual para otros documentos del corpus, así que no tiene una única respuesta correcta.

Las consultas que no se descartan se usan tal como las escribió el LLM.

In [9]:
for c in candidates.itertuples():
    print(f'━━━ {c.cid} · {c.title} ({c.source}) · {c.chunk_id}')
    print(f'CONSULTA:  {c.query}')
    print(f'FRAGMENTO: {" ".join(c.text.split())}\n')

━━━ c001 · Eutelsat 33E (Wikipedia 2000–2025) · doc_0488__0002
CONSULTA:  French satellite carrying military payloads launched from Guiana Space Centre on February 12, 2009 with Ariane rocket
FRAGMENTO: lb) and is expected to operate for 15 years. The spacecraft has 64 Ku-band transponders , broadcasting satellite television and radio to Europe, North Africa and the Middle East. Hot Bird 10, as it was then named, was launched by Arianespace using an Ariane 5ECA carrier rocket, as part of a dual launch with the NSS-9 spacecraft. Two French military satellites, Spirale-A and Spirale-B were also carried, as secondary payloads. The launch took place from ELA-3 at Kourou , French Guiana , at 22:09 UTC on 12 February 2009. The spacecraft was deployed into geosynchronous transfer orbit , raising itself to its operational geostationary orbit by means of its apogee motor . Upon its entry into service, prior to becoming part of Eutelsat's Hot Bird fleet, the Hot Bird 10 satellite was used to cov

**Resultado de la revisión.** Se apunta en `DESCARTES` cada consulta descartada con su motivo, que empieza por `trivial` o `ambigua`. Cuando la revisión está completa, se pone `REVISION_TERMINADA = True` y se vuelve a ejecutar el notebook: entonces se escribe `queries.jsonl` y se calculan los baselines.

In [10]:
REVISION_TERMINADA = True
DESCARTES = {
    'c002': 'ambigua: da a entender que aterrizó por un motivo clasificado, pero el texto solo dice que el objetivo de la misión era secreto',
    'c004': 'ambigua: en julio de 1969 falló un instrumento y la misión siguió operando hasta 1972',
    'c008': 'ambigua: parece que hubo dos visitas, pero la tripulación llegó en marzo y se quedó hasta mayo',
    'c017': 'ambigua: da a entender que el satélite iba destinado a la ISS, pero se desplegó desde el transbordador',
    'c019': 'ambigua: dice que seguía operativo tras 18 años, pero el texto indica que se retiró en 2013',
    'c026': 'ambigua: STS-121 se lanzó en julio de 2006 y la consulta dice 2007',
    'c027': 'ambigua: la astronauta coreana iba en el lanzamiento de abril y el pasajero estadounidense en el regreso de octubre',
    'c037': 'ambigua: mezcla los problemas del escáner con el fallo a baja altitud, que fue de otro instrumento',
    'c039': 'ambigua: marzo de 2022 es la fecha de la foto previa al vuelo, no la del lanzamiento',
    'c043': 'trivial: copia casi palabra por palabra la frase que identifica a AfriStar por su emisión de radio digital',
    'c062': 'ambigua: dice que explotó el satélite, pero lo que explotó fue el cohete que lo llevaba',
    'c069': 'ambigua: sitúa la pérdida de la carga dos años después, cuando ocurrió unos ocho meses tras el lanzamiento',
    'c075': 'ambigua: habla de claridad del agua en general y el fragmento estudia la detección de cambios en la clorofila',
    'c078': 'ambigua: Apollo 6 voló en 1968 y la trayectoria más allá de la Luna quedó en el plan de vuelo',
    'c081': 'ambigua: da a entender que la nave fue a la ISS para el experimento, pero la prueba del cable se intentó después de separarse',
    'c085': 'ambigua: OV1-17 se lanzó en 1969 y la consulta dice 1966',
    'c086': 'ambigua: presenta la reentrada como un fallo, pero el fragmento solo dice que el satélite perdió órbita',
    'c098': 'ambigua: tras fallar la baliza se recurrió a cámaras de seguimiento, no al método que indica la consulta',
}

In [11]:
unknown = set(DESCARTES) - set(candidates.cid)
assert not unknown, f'Identificadores que no existen: {sorted(unknown)}'
assert all(m.split(':')[0].strip() in ('trivial', 'ambigua') for m in DESCARTES.values()), \
    'Cada motivo debe empezar por "trivial" o "ambigua"'

candidates['decision'] = candidates.cid.map(lambda c: 'descartada' if c in DESCARTES else 'conservada')
candidates['motivo'] = candidates.cid.map(lambda c: DESCARTES.get(c, ''))
kept = candidates[candidates.decision == 'conservada'].reset_index(drop=True)
kept['qid'] = [f'q{i:02d}' for i in range(1, len(kept) + 1)]

print(f'Revisadas: {len(candidates)} consultas · descartadas: {len(DESCARTES)} '
      f'({Counter(m.split(":")[0].strip() for m in DESCARTES.values())}) · conservadas: {len(kept)}')

if REVISION_TERMINADA:
    assert len(kept) >= 30, 'El enunciado pide al menos 30 consultas'
    with QUERIES_PATH.open('w', encoding='utf-8', newline='\n') as f:
        for q in kept.itertuples():
            f.write(json.dumps({'qid': q.qid, 'query': q.query, 'relevant_ids': [q.doc_id], 'source_chunk': q.chunk_id},
                               ensure_ascii=False) + '\n')
    print(f'{len(kept)} consultas guardadas en {QUERIES_PATH}')
else:
    print('Revisión pendiente: queries.jsonl no se escribe todavía.')

candidates.loc[candidates.decision == 'descartada', ['cid', 'title', 'query', 'motivo']]

Revisadas: 100 consultas · descartadas: 18 (Counter({'ambigua': 17, 'trivial': 1})) · conservadas: 82
82 consultas guardadas en queries.jsonl


,cid,title,query,motivo
1,c002,OTV-6,US military classified satellite launched from Cape Canaveral in May 2020 that landed after near...,"ambigua: da a entender que aterrizó por un motivo clasificado, pero el texto solo dice que el ob..."
3,c004,Nimbus 3,Spacecraft with infrared spectrometer and radioisotope power source that operated until July 196...,ambigua: en julio de 1969 falló un instrumento y la misión siguió operando hasta 1972
7,c008,Soyuz T-4,Crew visited Soviet space station in March and May 1981 with a unique rescue vehicle experiment ...,"ambigua: parece que hubo dos visitas, pero la tripulación llegó en marzo y se quedó hasta mayo"
16,c017,STS-96,Space shuttle carrying a unique reflective satellite to ISS in 1999 with international student o...,"ambigua: da a entender que el satélite iba destinado a la ISS, pero se desplegó desde el transbo..."
18,c019,Nilesat 101,"Egyptian geosynchronous communications satellite launched in April 1998 from Kourou, still opera...","ambigua: dice que seguía operativo tras 18 años, pero el texto indica que se retiró en 2013"
25,c026,STS-121,"Shuttle carrying astronauts with families sending personalized music messages, launched from Ken...",ambigua: STS-121 se lanzó en julio de 2006 y la consulta dice 2007
26,c027,Soyuz TMA-12,Russian space crew that included first Korean astronaut in April 2008 with a private American pa...,ambigua: la astronauta coreana iba en el lanzamiento de abril y el pasajero estadounidense en el...
36,c037,Explorer 6,Spacecraft with TV scanner that failed due to poor orientation and logic circuit issues in Augus...,"ambigua: mezcla los problemas del escáner con el fallo a baja altitud, que fue de otro instrumento"
38,c039,SBUDNIC,Brown University CubeSat launched in March 2022 with Arduino and AA batteries that decayed from ...,"ambigua: marzo de 2022 es la fecha de la foto previa al vuelo, no la del lanzamiento"
42,c043,AfriStar,Geostationary communications satellite launched by Ariane in October 1998 that broadcast digital...,trivial: copia casi palabra por palabra la frase que identifica a AfriStar por su emisión de rad...


## **5. Baselines: TF-IDF + coseno y BM25**

Como en la sección 4 de *Text Vectorization I*:

* **TF-IDF + similitud coseno** con Gensim: `TfidfModel` con sus valores por defecto (peso $\text{tf} \cdot \log_2(N/\text{df})$ y vectores de norma 1) y `SparseMatrixSimilarity` para la similitud coseno.
* **BM25** con la librería `bm25s`, variante **Lucene**, con $k_1 = 1.5$ y $b = 0.75$:

$$\text{BM25}(q, d) = \sum_{t \in q} \ln\left(1 + \frac{N - \text{df}(t) + 0.5}{\text{df}(t) + 0.5}\right) \frac{f(t,d)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

$N$, $\text{df}$, $|d|$ y avgdl se calculan sobre **fragmentos**. Las consultas se preprocesan con la misma función `preprocess` que los fragmentos.

**Ranking de documentos.** Se puntúan todos los fragmentos, cada documento recibe la puntuación de su **mejor fragmento**, se eliminan los documentos repetidos y se quedan los **10 primeros**. Un fragmento con puntuación 0 no comparte ningún término con la consulta y no cuenta como recuperado.

In [12]:
%%time
corpus_bow = [D.doc2bow(tokens) for tokens in frags.tokens]
tfidf = TfidfModel(corpus_bow)
tfidf_index = SparseMatrixSimilarity(tfidf[corpus_bow], num_features=len(D))

bm25 = bm25s.BM25(k1=CONFIG['bm25_k1'], b=CONFIG['bm25_b'], method=CONFIG['bm25_method'])
bm25.index(frags.tokens.tolist(), show_progress=False)

def scores_tfidf(tokens):
    return np.asarray(tfidf_index[tfidf[D.doc2bow(tokens)]])

def scores_bm25(tokens):
    known = [t for t in tokens if t in bm25.vocab_dict]
    return bm25.get_scores(known) if known else np.zeros(len(frags))

def rank_documents(scores, k=CONFIG['top_k']):
    ranking = []
    for i in np.argsort(-scores, kind='stable'):        # fragmentos de mayor a menor puntuación
        if scores[i] <= 0 or len(ranking) == k:
            break
        if frags.doc_id[i] not in ranking:              # cada documento, con su mejor fragmento
            ranking.append(frags.doc_id[i])
    return ranking

print(f'Índices construidos sobre {len(frags):,} fragmentos; longitud media {n_tokens.mean():.1f} tokens')

Índices construidos sobre 19,620 fragmentos; longitud media 131.3 tokens
CPU times: total: 7.17 s
Wall time: 7.32 s


In [13]:
if QUERIES_PATH.exists():
    queries = pd.read_json(QUERIES_PATH, lines=True, dtype={'qid': str})
    queries['tokens'] = preprocess(queries['query'].tolist())
    rankings = {'TF-IDF': {q.qid: rank_documents(scores_tfidf(q.tokens)) for q in queries.itertuples()},
                'BM25': {q.qid: rank_documents(scores_bm25(q.tokens)) for q in queries.itertuples()}}
    titles = docs.set_index('doc_id').title
    example = queries.iloc[0]
    print(f'Consulta {example.qid}: {example.query}')
    print(f'Tokens: {example.tokens}')
    print(f'Documento relevante: {titles[example.relevant_ids[0]]}\n')
    top = {method: [titles[d] for d in r[example.qid]] for method, r in rankings.items()}
    display(pd.DataFrame({m: t + [''] * (CONFIG['top_k'] - len(t)) for m, t in top.items()}, index=range(1, CONFIG['top_k'] + 1)))
else:
    print('Todavía no hay queries.jsonl: terminad la revisión de la sección 4.3.')

Consulta q01: French satellite carrying military payloads launched from Guiana Space Centre on February 12, 2009 with Ariane rocket
Tokens: ['french', 'satellite', 'carry', 'military', 'payload', 'launch', 'guiana_space', 'centre', 'february', '12', '2009', 'ariane', 'rocket']
Documento relevante: Eutelsat 33E



,TF-IDF,BM25
1,Starlette and Stella,Viking (satellite)
2,Viking (satellite),Starlette and Stella
3,Hélios 1B,Hélios 1B
4,Sentinel-1D,Composante Spatiale Optique
5,Pirs (ISS module),Eutelsat 4A
6,XMM-Newton,XMM-Newton
7,Cerise (satellite),Sentinel-1D
8,Tiba 1,Eutelsat 33E
9,Clémentine (satellite),Tiba 1
10,Ekspress-AM44,Cerise (satellite)
